In [1]:
import pandas as pd
import os.path
import argparse
import logging
import sys

covfile = '/scratch/mbachma3/lc_imputation_nf/results/2.mapped/M022074_192_L6_coverage.tsv'
samplefile = '/scratch/mbachma3/lc_imputation_nf/results/2.mapped/M022074_192_L6_sample_sex.tsv'

# get individual's name
indname = os.path.basename(covfile).replace('_coverage.tsv','')
# load df
cov = pd.read_csv(covfile, sep = "\t", header = 0)

# sex chromosome (the one without PAR)
scaffold13_depth = cov[cov['#rname'] == 'Super-Scaffold_13']['meandepth'].mean()

# filter out sex chromosome and chromosome with PAR
cov_autosomal = cov[~cov['#rname'].isin(['Super-Scaffold_13', 'Super-Scaffold_42'])]
autosomal_depth = cov_autosomal['meandepth'].mean()

# sex chromosome / automosome meandepth ratio
depth_ratio = scaffold13_depth/autosomal_depth

if depth_ratio > 0.65: 
    indsex = 'M'
else: 
    indsex = 'F'

print(f"{indname}\t{indsex}")

with open(samplefile, "w") as f:
    f.write(f"{indname}\t{indsex}")


M022074_192_L6	F


In [ ]:
import pandas as pd
import os.path
import argparse
import logging
import sys

# Individuals of interest
survival = os.listdir("/work/FAC/FBM/DEE/jgoudet/barn_owl/Common/survival/1.data/1.fq/")

survival_df = pd.DataFrame(survival)
ring_ids =survival_df[0].str.split('_').str[0].drop_duplicates().to_frame('RingId')
ring_ids

# full metadata table
db = pd.read_csv("/scratch/mbachma3/lc_imputation_nf/data/BarnOwls_Legacy_20231010153920.csv", sep =',', header = 0)

survival_db = db[db['RingId'].isin(ring_ids['RingId'])].reset_index()

print(db.shape)
print(ring_ids.shape)
print(survival_db.shape)

# We are missing about 38 individuals present in Survival, absent in Database 
# After discussing with Anna it turns out any NB individuals are the ones that died before getting ringed
# Do not expect t'NB' to be in the owl database.

survival_sample_sex = survival_db[["RingId", "PhenotypeSex"]]
males = survival_sample_sex[survival_sample_sex["PhenotypeSex"] == "Male"]
females = survival_sample_sex[survival_sample_sex["PhenotypeSex"] == "Female"]
unknown = survival_sample_sex[survival_sample_sex["PhenotypeSex"].isna()]

print(males.shape)
print(females.shape)
print(unknown.shape)

(13311, 32)
(714, 1)
(676, 33)


In [ ]:
# Write 
males['RingId'].sample(n=5).to_csv('/scratch/mbachma3/lc_imputation_nf/data/pheno_male_subsample.txt', header=None, index=None)
females['RingId'].sample(n=5).to_csv('/scratch/mbachma3/lc_imputation_nf/data/pheno_female_subsample.txt', header=None, index=None)